# 05 Imaging Denoising Pipeline

Configure and run ANN-based denoising for imaging data.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:
import os
import sys

# change to the upper level folder to detect dj_local_conf.json
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')

import datajoint as dj
from adamacs.pipeline import subject, session, equipment, surgery, event, trial, imaging, behavior, scan, model,  analysis, denoising
from adamacs.ingest import session as isess
from adamacs.ingest import behavior as ibe
from adamacs.utility import *
from adamacs.helpers import stack_helpers as sh
import re
import numpy as np

dj.__version__

print(dj.__version__)
print(dj.config['custom']['database.prefix'])

 
# Function to find the file with the maximum iteration number in the given directory
def find_max_iteration_file(directory):
    files = [
        f for f in os.listdir(directory)
        if (match := re.match(r"model_(\d+)\.pth", f))
    ]
    return os.path.join(directory, max(files, key=lambda x: int(re.search(r"model_(\d+)\.pth", x).group(1)))) if files else None



# Pipeline integration


Enter Model Parameters

In [ ]:
denoisemodel_dir = dj.config['custom'].get('support_denoising_model_dir')[0]
# model_file = 'scan9FRQO5LP_NK_ROS-1936_02033.tif'
# model_file = 'scan9FRGLKR0_NK_ROS-1936_02025.tif' #Natasah big
model_file = 'scan9FSABQM4_JJ_ROS-1938_02030.tif' #Jisoo big
model_file = 'scan9FRFUTVB_NK_ROS-1936_02036.tif'
# model_file = 'scan9FS7X4NO_LE_ROS-1985_00001.tif'
# output_file = '/datajoint-data/models/tobiasr/SUPPORT/scan9FRQO5LP_NK_ROS-1936_02033.tif'


In [ ]:
modelout = os.path.join(denoisemodel_dir, "saved_models", model_file)

model_file_max = find_max_iteration_file(modelout)
print(f"The file with the maximum iteration is: {model_file_max}" if model_file_max else "No matching files found.")
print(denoisemodel_dir)
# model_file_max = '/datajoint-data/models/tobiasr/SUPPORT/saved_models/scan9FRGLKR0_NK_ROS-1936_02025.tif/model_150.pth'

In [ ]:
# params_denoise = {
#     "model_file": model_file_max,
#     "output_file": modelout,
#     "patch_size": [61, 64, 64],      # WAIT! We trained to 128 or not?
#     "patch_interval": [1, 32, 32],
#     "batch_size": 1000,  # Lower it if memory exceeds
#     "bs_size": 3,  # Modify if you changed bs_size when training
#     "bp_mode": False,
#     # "device": device,
#     "unet_channels": [16, 32, 64, 128, 256],  # same as mid_channels
#     "depth": 5,
#     "blind_conv_channels": 64,
#     "one_by_one_channels": [32, 16],
#     "last_layer_channels": [64, 32, 16]
# }


In [ ]:
# NEW BIG MODEL TRAINING Natasha / Jisoo
# Training options used: 
# Namespace(batch_size=16, blind_conv_channels=64, bp=False, bs_size=[3, 3], checkpoint_interval=5, depth=5, epoch=0, exp_name='scan9FRGLKR0_NK_ROS-1936_02025.tif', input_frames=61, is_folder=False, 
# last_layer_channels=[64, 32, 16], logging_interval=1, logging_interval_batch=50, loss_coef=[0.5, 0.5], lr=0.0005, n_cpu=8, n_epochs=200, noisy_data=['/datajoint-data/data/nataliak/NK_ROS-1936_2024-12-03_scan9FRGLKR0_sess9FRGLKR0/scan9FRGLKR0_NK_ROS-1936_02025.tif'], 
# one_by_one_channels=[32, 16], 
# patch_interval=[1, 64, 64], patch_size=[61, 128, 128], random_seed=0, results_dir='/datajoint-data/models/tobiasr/SUPPORT', sample_interval=10, sample_max_t=600, 
# unet_channels=[64, 128, 256, 512, 1024], use_CPU=False)

params_denoise = {
    "model_file": model_file_max,
    "output_file": modelout,
    "patch_size": [61, 128, 128],      
    "patch_interval": [1, 64, 64],    #if blocky reduce this - otherwise half patch
    "batch_size": 1000,  # Lower it if memory exceeds
    "bs_size": 3,  # Modify if you changed bs_size when training
    "bp_mode": False,
    # "device": device,
    "unet_channels": [64, 128, 256, 512, 1024],  # same as mid_channels
    "depth": 5,
    "blind_conv_channels": 64,
    "one_by_one_channels": [32, 16],
    "last_layer_channels": [64, 32, 16]
}


In [ ]:
modelout

In [ ]:
# paramsetlname = (denoising.DenoisingTask & scan_key & 'denoise_paramset_idx = 20').fetch1('denoise_paramset_idx')
# denoisename = (denoising.DenoisingProcessingParamSet & f"denoise_paramset_idx = {paramsetlname}").fetch1('denoise_paramset_desc')
# params = (denoising.DenoisingProcessingParamSet & f"denoise_paramset_idx = {paramsetlname}").fetch1('denoise_params')
# model_file_path = params['model_file']
# last_two_dirs = os.path.join(*model_file_path.split(os.sep)[-2:-1])
# modelfilename = f"{last_two_dirs}_{os.path.basename(model_file_path)}"


In [ ]:
(denoising.DenoisingProcessingParamSet & "denoise_paramset_idx = 25").delete()
denoising.DenoisingProcessingParamSet.insert_new_params(
    denoising_method="support",
    # concatenation_method="indiv",
    paramset_idx = 13,      # The index of the Imaging parameter set to use for registration
    denoise_paramset_idx=30,
    denoise_params=params_denoise,
    denoise_paramset_desc="TR: Support, mini2p 122it model BIG scan9FRFUTVB_NK_ROS-1936_02036"
)


In [ ]:
denoising.DenoisingProcessingParamSet()

Add denoising task 

In [ ]:
scansi = "scan9FRFD943"
scan_key = (scan.Scan() & f"scan_id = '{scansi}'").fetch1("KEY")
# (denoising.DenoisingTask & scan_key).delete()

In [ ]:
channels = (scan.ScanInfo & scan_key).fetch1('nchannels')
channels


In [ ]:
scan_filepath = (scan.ScanPath() & scan_key).fetch1('path')

denoising.DenoisingTask.insert1({
    'session_id': scan_key['session_id'],
    'scan_id': scan_key['scan_id'],
    # 'paramset_idx': 13,  # Replace 'some_field' with the correct field name
    'denoise_paramset_idx': 30,  # Replace 'some_field' with the correct field name
    'processing_output_dir': scan_filepath,  # Ensure this matches the correct field name
    'task_mode': 'trigger'  # Replace with the appropriate field name
}, skip_duplicates=True)

In [ ]:
denoising.DenoisingTask & scan_key

In [ ]:
print((scan.ScanPath() & scan_key).fetch1('path'))

In [ ]:
denoising.DenoisingTask & scan_key


In [ ]:
denoising.Denoising & scan_key

In [ ]:
denoising.Denoising() & 'denoise_paramset_idx = 30'

In [ ]:
populate_settings = {'display_progress': True, 'suppress_errors': False, 'processes': 1}
# try:
# imaging.Processing.populate(*restriction, **populate_settings)
denoising.Denoising.populate(scan_key, **populate_settings)

In [ ]:
(imaging.Processing() & scan_key).delete()

In [ ]:
scans_to_process = denoising.Denoising * imaging.Processing & 'denoise_paramset_idx = 30' & 'paramset_idx = 1060'
scans_to_process

In [ ]:
imaging.Processing & scan_key

In [ ]:
 (denoising.DenoisingTask & scan_key & 'denoise_paramset_idx = 30')

In [ ]:
scans_to_process

In [ ]:
#make movie for inspection

import skimage.io as skio
import glob

for scan_key in scans_to_process:

    output_dir = (denoising.DenoisingTask & scan_key & 'denoise_paramset_idx = 30').fetch1("processing_output_dir") + '/' + 'support'

    mp4_files = glob.glob(os.path.join(output_dir, "*denoised*.tif"))
    output_file = mp4_files[0] if mp4_files else None

    # output_file = "/datajoint-data/data/nataliak/NK_ROS-1936_2024-12-20_scan9FRQO5LP_sess9FRQO5LP/support/scan9FRQO5LP_NK_ROS-1936_02033_denoised_stack.tif"
    finalstack_padded = skio.imread(output_file)
    sh.make_stack_movie(finalstack_padded, output_file + "scan9FRFUTVB_NK_ROS-1936_02036_model_128.pth.mp4", 120, p1set=0.5, p2set=99.98)
    print((scan.ScanPath() & scan_key).fetch1('path'))


In [ ]:
scansi = "scan9FRFD943"
scan_key = (scan.Scan() & f"scan_id = '{scansi}'").fetch1("KEY")
# (denoising.DenoisingTask & scan_key).delete()
scan_key

In [ ]:
paramsetlname = (denoising.DenoisingTask & scan_key & 'denoise_paramset_idx = 30').fetch1('denoise_paramset_idx')
denoisename = (denoising.DenoisingProcessingParamSet & f"denoise_paramset_idx = {paramsetlname}").fetch1('denoise_paramset_desc')
params = (denoising.DenoisingProcessingParamSet & f"denoise_paramset_idx = {paramsetlname}").fetch1('denoise_params')
model_file_path = params['model_file']
last_two_dirs = os.path.join(*model_file_path.split(os.sep)[-2:-1])
modelfilename = f"{last_two_dirs}_{os.path.basename(model_file_path)}"


In [ ]:
output_file

In [ ]:
output_file + modelfilename

In [ ]:
#SINGLE make movie for inspection

import skimage.io as skio
import glob

output_dir = (denoising.DenoisingTask & scan_key & 'denoise_paramset_idx = 15').fetch1("processing_output_dir") + '/' + 'support'

mp4_files = glob.glob(os.path.join(output_dir, "*denoised*.tif"))
output_file = mp4_files[0] if mp4_files else None

# output_file = "/datajoint-data/data/nataliak/NK_ROS-1936_2024-12-20_scan9FRQO5LP_sess9FRQO5LP/support/scan9FRQO5LP_NK_ROS-1936_02033_denoised_stack.tif"
finalstack_padded = skio.imread(output_file)
sh.make_stack_movie(finalstack_padded, output_file + modelfilename, fpsset=120, p1set=0.5, p2set=99.98)
print((scan.ScanPath() & scan_key).fetch1('path'))


In [ ]:
import subprocess
import os
import glob
from pathlib import Path

# Optional destination SSH info
local_mac_user = os.environ.get('LOCAL_MAC_USER', 'your_user')
local_mac_ip = os.environ.get('LOCAL_MAC_IP', 'localhost')

# Define your local directory for downloads
local_temp_dir = Path(os.environ.get('LOCAL_TEMP_DIR', str(Path.home() / 'temp')))
local_temp_dir.mkdir(parents=True, exist_ok=True)

# Define the source path on the remote server
scan_filepath = '/path/to/scan'  # Update this to the correct remote path
source_path = os.path.join(scan_filepath, 'support')
denoised_files = glob.glob(os.path.join(source_path, '*denoised*.tif'))

if denoised_files:
    denoised_file = denoised_files[0]
    destination_path = local_temp_dir / os.path.basename(denoised_file)

    rsync_command = [
        'rsync',
        '-avz',
        f'{denoised_file}',
        f'{local_mac_user}@{local_mac_ip}:{destination_path}',
    ]

    subprocess.run(rsync_command)

    print(f'Denoised file downloaded to: {destination_path}')
else:
    print('No denoised .tif file found.')

In [ ]:
denoising.Denoising & scan_key

Ppopulation will be done on the GPU server - not locally!

# Example for bulk denoising

In [ ]:
# scan.ScanInfo.ScanFile & userkey & "file_path LIKE '%dummy%'"

scans_to_process = (session.Session * scan.ScanInfo.ScanFile * session.SessionUser * subject.User * event.BehaviorRecording * scan.ScanPath 
                    & "session_datetime >= '2024-08-02'" & "initials = 'JJ'" & "file_path NOT LIKE '%dummy%'").fetch("KEY")

scans_to_process 


In [ ]:

scansi = "scan9FRQO5LP"
scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch('KEY')
scans_to_process = (scan.ScanPath() * denoising.Denoising & scan_key).fetch('KEY')


In [ ]:
(denoising.Denoising & scan_key).delete()

In [ ]:
# scans_to_process
for scan_key in scans_to_process:
    scan_filepath = (scan.ScanPath() & scan_key).fetch1('path')
    print(scan_filepath)
    denoising.DenoisingTask.insert1({
        'session_id': scan_key['session_id'],
        'scan_id': scan_key['scan_id'],
        # 'paramset_idx': 13,  # Replace 'some_field' with the correct field name
        'denoise_paramset_idx': 30,  # Replace 'some_field' with the correct field name
        'processing_output_dir': scan_filepath,  # Ensure this matches the correct field name
        'task_mode': 'trigger'  # Replace with the appropriate field name
    }, skip_duplicates=True)


In [ ]:
print((denoising.DenoisingTask() & scan_key).fetch1("processing_output_dir"))

In [ ]:
(denoising.Denoising & scan_key).delete()


In [ ]:
populate_settings = {'display_progress': True, 'suppress_errors': False, 'processes': 1}
# try:
# imaging.Processing.populate(*restriction, **populate_settings)
denoising.Denoising.populate(scans_to_process, **populate_settings)

In [ ]:
mdl = dj.schema('roselab_' + 'denoising')
display(mdl.jobs)
# mdl.jobs.delete()
# (mdl.jobs & 'status="error"').delete()
# (mdl.jobs & 'status="error"').fetch('key'

### bulk post-hoc s2p on same data

In [ ]:

scansi = "scan9FRQO5LP"
scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch('KEY')
scans_to_process = (scan.ScanPath() * denoising.Denoising & scan_key).fetch('KEY')


In [ ]:
scansi = "scan9FRFD943"
scan_key = (scan.Scan() & f"scan_id = '{scansi}'").fetch1("KEY")
# (denoising.DenoisingTask & scan_key).delete()

In [ ]:
scans_to_process = (session.Session * session.SessionUser * subject.User * event.BehaviorRecording * scan.ScanPath & scan_key).fetch("KEY")
# scans_to_process

In [ ]:
scans_to_process

In [ ]:
(imaging.ProcessingTask & scan_key).delete()

In [ ]:
(imaging.ProcessingParamSet & "paramset_idx = 1060").fetch1("params")

In [ ]:

# existing_data = (scan.Scan * scan.ScanPath & prockey)
selected_s2pparms_index = 1060

for record in scans_to_process:
    # record['path'] = Path(str(record['path']).replace("tobiasr", "nataliak"))
    # print(record['path'])
    imaging.ProcessingTask.insert1((record['session_id'], record['scan_id'], selected_s2pparms_index, record['path'], 'trigger'), skip_duplicates=True)

In [ ]:

# existing_data = (scan.Scan * scan.ScanPath & prockey)
selected_s2pparms_index = 1060

for record in [scan_key]:
    # record['path'] = Path(str(record['path']).replace("tobiasr", "nataliak"))
    # print(record['path'])
    imaging.ProcessingTask.insert1((record['session_id'], record['scan_id'], selected_s2pparms_index, record['path'], 'trigger'), skip_duplicates=True)

In [ ]:
imaging.Processing & scan_key

In [ ]:
imaging.Fluorescence.Trace & scan_key & 'paramset_idx = 1060'

In [ ]:
imaging.ProcessingParamSet()

In [ ]:
 (imaging.ProcessingTask & scan_key & 'paramset_idx = 1060').delete()

In [ ]:

scansi = "scan9FRFD943"
scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch('KEY')
scans_to_process = (scan.ScanPath() * denoising.Denoising & scan_key).fetch('KEY')


In [ ]:
print((scan.ScanPath() & scan_key).fetch1('path'))

In [ ]:
scans_to_process = (session.Session * session.SessionUser * subject.User * event.BehaviorRecording * scan.ScanPath & "session_datetime >= '2024-10-02'" & "initials = 'NK'").fetch("KEY")

denoising_paraamset_idx = 30
scans_to_process = (denoising.Denoising * scan.ScanPath & scans_to_process & f"denoise_paramset_idx = {denoising_paraamset_idx}").fetch("KEY")
scans_to_process

In [ ]:

# existing_data = (scan.Scan * scan.ScanPath & prockey)
selected_s2pparms_index = 1060

for record in scans_to_process:
    # record['path'] = Path(str(record['path']).replace("tobiasr", "nataliak"))
    # print(record['path'])
    imaging.ProcessingTask.insert1((record['session_id'], record['scan_id'], selected_s2pparms_index, record['path'], 'trigger'), skip_duplicates=True)

In [ ]:
(imaging.ProcessingTask & scans_to_process & f"paramset_idx = {selected_s2pparms_index}").delete()

In [ ]:
#load the scan, session s2p parameter index from the proccessing table (the first s2p run)
key = (imaging.Processing & scan_key & f'paramset_idx = {selected_s2pparms_index}').fetch1('KEY')
# set manual curation to TRUE
manual_curation = False
# do a new Curation task
imaging.Curation().create1_from_processing_task(key, is_curated=manual_curation)

In [ ]:
populate_settings = {'display_progress': True, 'suppress_errors': True, 'processes': 1}

imaging.MotionCorrection.populate(**populate_settings)

imaging.Segmentation.populate(**populate_settings)

imaging.MaskClassification.populate(**populate_settings)

imaging.Fluorescence.populate(**populate_settings)

imaging.Activity.populate(**populate_settings)

In [ ]:
(imaging.ProcessingTask() & scans_to_process & f'paramset_idx={selected_s2pparms_index}')

In [ ]:
scan_key = imaging.ProcessingTask() & "scan_id = 'scan9FRQO5LP'"
imaging.ProcessingTask() & "scan_id = 'scan9FRQO5LP'"

In [ ]:
(denoising.Denoising & scan_key).delete()

In [ ]:
populate_settings = {'display_progress': True, 'suppress_errors': True, 'processes': 1}
# try:
# imaging.Processing.populate(*restriction, **populate_settings)
denoising.Denoising.populate(**populate_settings)

In [ ]:
imaging.ProcessingTask() & "paramset_idx > 1000"

In [ ]:
# populate_settings = {'display_progress': True, 'suppress_errors': False, 'processes': 1}
# # try:
# # imaging.Processing.populate(*restriction, **populate_settings)
# denoising.Denoising.populate(**populate_settings)